# Notebook 1 — Where does each PEFT method operate?

This notebook is a **minimal, visual intuition builder** for a handful of PEFT families.

We keep a tiny frozen transformer-like block fixed and ask:

- **Prompt / prefix methods:** how do trainable tokens modify the **input space**?
- **Adapters:** how do residual bottlenecks modify the **hidden-state / activation path**?
- **LoRA:** how do low-rank matrices modify the **weight space**?
- **BitFit:** what happens when we touch **only biases**?
- **Linear probing:** what if the encoder is frozen and only the **readout head** changes?

This notebook is intentionally small and pedagogical. It is designed to help participants form the right mental model before running larger experiments.

In [ ]:
# Optional install cell for Colab / fresh environments
# !pip install -q torch matplotlib pandas

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1) A tiny frozen block

We will use a tiny transformer-style block with:

- input embeddings `x`
- a single self-attention-style projection path
- a feed-forward path
- layer norms and residual connections

We will **freeze the base block**, then create variants that add trainable state in different places.

In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, d_model=32, d_hidden=64):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_hidden),
            nn.GELU(),
            nn.Linear(d_hidden, d_model),
        )

    def forward(self, x):
        # x: [batch, seq, d_model]
        h = self.ln1(x)
        q = self.q(h)
        k = self.k(h)
        v = self.v(h)
        attn = (q @ k.transpose(-1, -2)) / math.sqrt(q.size(-1))
        attn = attn.softmax(dim=-1)
        x = x + self.o(attn @ v)

        h = self.ln2(x)
        x = x + self.ffn(h)
        return x


base_block = TinyBlock().to(device)
for p in base_block.parameters():
    p.requires_grad = False

sum(p.numel() for p in base_block.parameters())

## 2) A single shared input batch

We create one synthetic "mini-batch" of token embeddings to probe the frozen block.

In [ ]:
B, T, D = 8, 12, 32
x = torch.randn(B, T, D, device=device)

with torch.no_grad():
    y_base = base_block(x)

print("input shape:", x.shape)
print("output shape:", y_base.shape)

## 3) PEFT variants as small wrappers

Each wrapper below isolates **where** the trainable change enters the computation.

### A. Linear probing
The encoder stays frozen. Only a task head changes.

### B. Soft prompt tuning
We prepend trainable prompt embeddings to the input sequence.  
This changes the **input / token space** before the frozen model processes anything.

### C. Adapter
We insert a small trainable bottleneck in the residual stream.  
This changes the **hidden activations** without replacing the pretrained weights.

### D. LoRA
We add a low-rank update to a frozen weight matrix.  
This changes the **effective weights** seen by the forward pass.

### E. BitFit
We only expose bias terms as trainable.

In [ ]:
class FrozenEncoderWithHead(nn.Module):
    def __init__(self, encoder, d_model=32, n_classes=3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        h = self.encoder(x)
        pooled = h.mean(dim=1)
        return self.head(pooled)


class SoftPromptWrapper(nn.Module):
    def __init__(self, encoder, prompt_len=4, d_model=32):
        super().__init__()
        self.encoder = encoder
        self.prompt = nn.Parameter(torch.randn(1, prompt_len, d_model) * 0.02)

    def forward(self, x):
        B = x.size(0)
        prompt = self.prompt.expand(B, -1, -1)
        x_prompt = torch.cat([prompt, x], dim=1)
        h = self.encoder(x_prompt)
        return h[:, self.prompt.size(1):, :]  # return only original token positions


class Adapter(nn.Module):
    def __init__(self, d_model=32, bottleneck=8):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class AdapterWrapper(nn.Module):
    def __init__(self, encoder, d_model=32, bottleneck=8):
        super().__init__()
        self.encoder = encoder
        self.adapter = Adapter(d_model=d_model, bottleneck=bottleneck)

    def forward(self, x):
        h = self.encoder(x)
        return self.adapter(h)


class LoRALinear(nn.Module):
    def __init__(self, frozen_linear: nn.Linear, rank=4, alpha=8.0):
        super().__init__()
        self.frozen = frozen_linear
        self.rank = rank
        self.alpha = alpha
        self.A = nn.Parameter(torch.randn(frozen_linear.in_features, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, frozen_linear.out_features))

        for p in self.frozen.parameters():
            p.requires_grad = False

    def forward(self, x):
        base = self.frozen(x)
        delta = x @ self.A @ self.B
        return base + (self.alpha / self.rank) * delta


class TinyBlockWithLoRA(nn.Module):
    def __init__(self, base_block: TinyBlock, rank=4):
        super().__init__()
        self.base = base_block
        self.q_lora = LoRALinear(base_block.q, rank=rank)

    def forward(self, x):
        h = self.base.ln1(x)
        q = self.q_lora(h)  # LoRA only on q projection
        k = self.base.k(h)
        v = self.base.v(h)
        attn = (q @ k.transpose(-1, -2)) / math.sqrt(q.size(-1))
        attn = attn.softmax(dim=-1)
        x = x + self.base.o(attn @ v)

        h = self.base.ln2(x)
        x = x + self.base.ffn(h)
        return x


class BitFitTinyBlock(nn.Module):
    def __init__(self, d_model=32, d_hidden=64):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.q = nn.Linear(d_model, d_model, bias=True)
        self.k = nn.Linear(d_model, d_model, bias=True)
        self.v = nn.Linear(d_model, d_model, bias=True)
        self.o = nn.Linear(d_model, d_model, bias=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_hidden, bias=True),
            nn.GELU(),
            nn.Linear(d_hidden, d_model, bias=True),
        )

        # start from the frozen block's weights
        with torch.no_grad():
            self.q.weight.copy_(base_block.q.weight)
            self.k.weight.copy_(base_block.k.weight)
            self.v.weight.copy_(base_block.v.weight)
            self.o.weight.copy_(base_block.o.weight)

            self.ln1.weight.copy_(base_block.ln1.weight)
            self.ln1.bias.copy_(base_block.ln1.bias)
            self.ln2.weight.copy_(base_block.ln2.weight)
            self.ln2.bias.copy_(base_block.ln2.bias)

            self.ffn[0].weight.copy_(base_block.ffn[0].weight)
            self.ffn[0].bias.copy_(base_block.ffn[0].bias)
            self.ffn[2].weight.copy_(base_block.ffn[2].weight)
            self.ffn[2].bias.copy_(base_block.ffn[2].bias)

        for name, p in self.named_parameters():
            p.requires_grad = ("bias" in name)

    def forward(self, x):
        h = self.ln1(x)
        q = self.q(h)
        k = self.k(h)
        v = self.v(h)
        attn = (q @ k.transpose(-1, -2)) / math.sqrt(q.size(-1))
        attn = attn.softmax(dim=-1)
        x = x + self.o(attn @ v)

        h = self.ln2(x)
        x = x + self.ffn(h)
        return x

In [ ]:
variants = {
    "soft_prompt": SoftPromptWrapper(base_block).to(device),
    "adapter": AdapterWrapper(base_block).to(device),
    "lora_q": TinyBlockWithLoRA(base_block).to(device),
    "bitfit": BitFitTinyBlock().to(device),
    "linear_probe_head": FrozenEncoderWithHead(base_block).to(device),
}

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

pd.DataFrame(
    [{"method": name, "trainable_params": count_trainable_params(model)}
     for name, model in variants.items()]
).sort_values("trainable_params")

## 4) One-step representational comparison

We do not train yet.  
We only ask: **if I introduce a trainable mechanism here, which representation does it directly perturb?**

We compare each variant's output to the frozen baseline using:

- relative L2 shift
- mean cosine similarity

In [ ]:
def rep_stats(y_ref, y_new):
    flat_ref = y_ref.reshape(-1, y_ref.size(-1))
    flat_new = y_new.reshape(-1, y_new.size(-1))
    rel_l2 = (flat_new - flat_ref).norm(dim=-1).mean() / (flat_ref.norm(dim=-1).mean() + 1e-8)
    cos = F.cosine_similarity(flat_ref, flat_new, dim=-1).mean()
    return float(rel_l2.detach().cpu()), float(cos.detach().cpu())

rows = []
with torch.no_grad():
    y_ref = base_block(x)

    for name, model in variants.items():
        if name == "linear_probe_head":
            # head output lives in label space, so skip representation comparison
            continue
        y_new = model(x)
        rel_l2, cos = rep_stats(y_ref, y_new)
        rows.append({"method": name, "relative_L2_shift": rel_l2, "mean_cosine_to_base": cos})

df_shift = pd.DataFrame(rows).sort_values("relative_L2_shift")
df_shift

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(df_shift["method"], df_shift["relative_L2_shift"])
plt.ylabel("Relative L2 shift vs frozen output")
plt.title("Which PEFT change perturbs the representation most directly?")
plt.xticks(rotation=20)
plt.show()

## 5) Visual sketch of *where* the trainable state lives

This chart is intentionally schematic rather than mathematically exact.

In [ ]:
methods = ["soft prompt", "adapter", "LoRA", "BitFit", "linear probe"]
ypos = [4, 3, 2, 1, 0]

plt.figure(figsize=(10, 4))
for y, m in zip(ypos, methods):
    plt.hlines(y, 0, 10, linewidth=1.5)
    plt.text(-0.2, y, m, va="center", ha="right", fontsize=11)

# base pipeline
plt.text(1, 4.5, "input tokens", ha="center", fontsize=11)
plt.text(4, 4.5, "frozen encoder", ha="center", fontsize=11)
plt.text(8, 4.5, "task head", ha="center", fontsize=11)

# highlight intervention locations
plt.scatter([1], [4], s=300, marker="s")  # soft prompt
plt.scatter([4], [3], s=300, marker="s")  # adapter
plt.scatter([4], [2], s=300, marker="s")  # lora
plt.scatter([4], [1], s=300, marker="s")  # bitfit
plt.scatter([8], [0], s=300, marker="s")  # linear probe

plt.text(1, 4, "input-space\ntrainable tokens", ha="center", va="center", fontsize=9)
plt.text(4, 3, "hidden-state\nresidual module", ha="center", va="center", fontsize=9)
plt.text(4, 2, "weight-space\nlow-rank update", ha="center", va="center", fontsize=9)
plt.text(4, 1, "subset of params:\nbias only", ha="center", va="center", fontsize=9)
plt.text(8, 0, "readout only", ha="center", va="center", fontsize=9)

plt.xlim(-1, 10.5)
plt.ylim(-0.8, 4.8)
plt.axis("off")
plt.show()

## 6) A small synthetic training exercise

Below we build a tiny sequence classification task so participants can see how each intervention can succeed or fail on the *same* frozen encoder.

The synthetic labels depend on a pattern in the sequence. That lets us observe:

- when changing only the head is enough
- when reshaping the input with prompts helps
- when weight-domain changes like LoRA are more expressive

In [ ]:
def make_synthetic_dataset(n=512, seq_len=12, d_model=32, n_classes=3):
    x = torch.randn(n, seq_len, d_model)
    # Labels are based on frozen-encoder pooled features.
    with torch.no_grad():
        h = base_block(x.to(device)).cpu().mean(dim=1)
    W = torch.randn(d_model, n_classes)
    y = (h @ W).argmax(dim=-1)
    return x, y

X, y = make_synthetic_dataset()
X_train, y_train = X[:400].to(device), y[:400].to(device)
X_val, y_val = X[400:].to(device), y[400:].to(device)
print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(self, encoder, d_model=32, n_classes=3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        h = self.encoder(x)
        pooled = h.mean(dim=1)
        return self.head(pooled)


def build_method(method):
    if method == "linear_probe":
        model = SequenceClassifier(base_block).to(device)
        for p in model.encoder.parameters():
            p.requires_grad = False

    elif method == "soft_prompt":
        model = SequenceClassifier(SoftPromptWrapper(base_block)).to(device)

    elif method == "adapter":
        model = SequenceClassifier(AdapterWrapper(base_block)).to(device)

    elif method == "lora":
        model = SequenceClassifier(TinyBlockWithLoRA(base_block)).to(device)

    elif method == "bitfit":
        model = SequenceClassifier(BitFitTinyBlock()).to(device)
        for p in model.head.parameters():
            p.requires_grad = True

    else:
        raise ValueError(method)

    return model


def fit(method, epochs=40, lr=3e-3):
    model = build_method(method)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    history = []

    for epoch in range(epochs):
        model.train()
        logits = model(X_train)
        loss = F.cross_entropy(logits, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            train_acc = (model(X_train).argmax(dim=-1) == y_train).float().mean().item()
            val_logits = model(X_val)
            val_loss = F.cross_entropy(val_logits, y_val).item()
            val_acc = (val_logits.argmax(dim=-1) == y_val).float().mean().item()

        history.append({
            "epoch": epoch + 1,
            "method": method,
            "train_loss": float(loss.item()),
            "val_loss": float(val_loss),
            "train_acc": train_acc,
            "val_acc": val_acc,
            "trainable_params": count_trainable_params(model),
        })

    return pd.DataFrame(history)


methods = ["linear_probe", "soft_prompt", "adapter", "lora", "bitfit"]
runs = [fit(m, epochs=30) for m in methods]
hist = pd.concat(runs, ignore_index=True)
hist.tail()

In [ ]:
plt.figure(figsize=(8, 4))
for method, g in hist.groupby("method"):
    plt.plot(g["epoch"], g["val_acc"], label=method)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Synthetic task: same base encoder, different adaptation locations")
plt.legend()
plt.show()

## 7) Practical takeaway

The point is **not** that one method universally wins.

The point is to internalize **what is being changed**:

| Method | Trainable object | Where it acts | Mental model |
|---|---|---|---|
| Linear probing | classifier head | output / label space | "reuse features as-is" |
| Soft prompt tuning | prompt embeddings | input space | "steer the frozen model by changing what it sees" |
| Adapter | bottleneck residual | hidden-state path | "add a small specialist module" |
| LoRA | low-rank weight delta | weight space | "change how a layer computes, cheaply" |
| BitFit | biases only | parameter subset | "nudge pretrained behavior" |

### Rule-of-thumb hypotheses to test later

- **Linear probing** is the first baseline when you think the pretrained features are already close to your task.
- **Soft / prompt tuning** is attractive when you want many task-specific states while keeping the base weights fixed.
- **Adapters** are useful when you want modularity and clear insertion points.
- **LoRA** is a strong default when you want more expressivity than prompts or BitFit, but still far fewer trainable weights than full finetuning.
- **BitFit** is a tiny, cheap baseline for small or mild shifts.

In your workshop, this notebook can serve as the "**where do they operate?**" intuition pass before participants move to a real checkpoint and dataset.